# K-Means Cluster Analysis (The DIANA Approach)
## Finding the "Root Cause" of Diabetes: A Guide for Non-ML Specialists

Welcome! If you are a doctor, a clinician, or just someone who wants to understand how the DIANA Artificial Intelligence works under the hood, this notebook is for you.

### What is "Clustering"?
Imagine you have a giant jar of mixed coins and you want a machine to sort them. You don't tell the machine *what* a quarter or a penny is; you just tell it: *"Put similar-looking coins into 4 different piles."* That is what **Unsupervised Clustering** does. 

---

## The Groundbreaking Science (Why we are doing this)
For decades, medicine treated "Type 2 Diabetes" as one single disease. But in 2018, a famous scientist named Emma Ahlqvist published a paper in *The Lancet* proving that **Type 2 Diabetes is actually 4 different diseases** (called "Phenotypes" or "Subtypes"). 

If a doctor knows exactly *which* of the 4 subtypes you have, they know exactly which medicine will work best for you. Our goal in this notebook is to use AI to find those 4 hidden subtypes in our patient data.

---
### The "Check Engine Light" Rule (Very Important!)
Why don't we let the AI look at the patient's Blood Sugar (HbA1c / FBS)? 

Think of High Blood Sugar as the **"Check Engine Light"** on a car dashboard. If a mechanic just looks at the light, they know the car is broken, but they don't know *why*. 

If we let the AI look at Blood Sugar, it takes the lazy way out. It just sorts people into "Mild Diabetes" and "Severe Diabetes."

By deliberately **hiding** the Blood Sugar from the AI, we force the AI to open the hood of the car and look at the engine. It is forced to sort people by the *actual root causes* of the disease (Obesity, Cholesterol, Age). This is how we find the true Ahlqvist subtypes!

## 1. Setup & Loading the Tools
*(First, we load all the necessary math and chart-drawing tools into the computer.)*

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import sys
import os
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

# Setup the drawing canvas for our charts
plt.rcParams["figure.dpi"] = 150
plt.rcParams["font.size"] = 11

# Connect to the main DIANA AI brain
BASE_DIR = Path(os.getcwd()).resolve().parents[2]
if str(BASE_DIR) not in sys.path:
    sys.path.insert(0, str(BASE_DIR))

from Ian_ML.common.paths import NHANES_PROCESSED_ROOT
from Ian_ML.common.feature_constants import CLUSTER_FEATURES
from Ian_ML.training.clustering import assign_ahlqvist_labels

print("System ready! Tools loaded.")

## 2. Filtering the Patients (The Gatekeeper)

You can't study different types of a disease if your waiting room is full of perfectly healthy people! 

Before we cluster, we run a filter. We explicitly remove the "Normal" (healthy) patients and **only** look at the people who are "At-Risk" or currently Diabetic. This ensures our 4 subtype buckets aren't skewed by healthy biology.

In [ ]:
# Load the patient spreadsheet
DATA_PATH = str(NHANES_PROCESSED_ROOT / "diana_dataset_final.csv")
df = pd.read_csv(DATA_PATH)

print(f"Total Patients in the Database: {df.shape[0]}")

# THE GATEKEEPER: Only keep patients who are "At-Risk" or Diabetic (Levels 1 and 2)
df['is_at_risk'] = (df['diabetes_label'] >= 1).astype(int)

df_at_risk = df[df['is_at_risk'] == 1].copy()
print(f"Sick/At-Risk Patients (We will study THESE people): {len(df_at_risk)}")

# Remove anyone who didn't take their blood tests
df_clean = df_at_risk.dropna(subset=CLUSTER_FEATURES).copy()
print(f"Final Cohort (Perfect data): {len(df_clean)} patients")

## 3. Prepping the Data (Apples to Apples)

If we tell the AI to compare Age (which goes from 0-100) and Triglycerides (which can go from 50 to 800), the math breaks because the numbers are too big and different.

We use a tool called a **Standard Scaler**. It mathematically shrinks and stretches every lab test so they are all on the exact same playground (measured in "Z-Scores"). Now, Age and Cholesterol can be compared fairly!

In [ ]:
# Grab only the safe lab tests (NO BLOOD SUGAR ALLOWED!)
X_raw = df_clean[CLUSTER_FEATURES].copy()

print("Notice what the AI is allowed to see (Age, BMI, Cholesterol):\n")
display(X_raw.head())

# Mathematically shrink the data so it's fair (Standardization)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_raw)

print("\n--- Data after the Standard Scaler (Apples to Apples) ---")
X_scaled_df = pd.DataFrame(X_scaled, columns=CLUSTER_FEATURES, index=X_raw.index)
display(X_scaled_df.describe().round(2))

## 4. The AI Does Its Job (K-Means Clustering)

Now we unleash the algorithm! We tell the AI: *"Please sort these patients into exactly 4 piles based on their biological similarities."*

After it makes the 4 piles, we use medical formulas to figure out what those piles actually represent. For example, if one pile has people with enormous Triglycerides and wide waists, we label them **SIRD** (Severe Insulin-Resistant).

In [ ]:
# Tell the AI to make 4 piles (K=4)
K = 4
kmeans = KMeans(n_clusters=K, random_state=42, n_init=10)
cluster_labels = kmeans.fit_predict(X_scaled) # "Go sort the patients!"

# Convert the scaled math back into real-world biology numbers (like pounds and mg/dL)
raw_centers = scaler.inverse_transform(kmeans.cluster_centers_)

# Give the clusters their official medical names based on their biology
label_map = assign_ahlqvist_labels(raw_centers, CLUSTER_FEATURES, K)

print("\nThe AI finished sorting! Here are the 4 medical subtypes it discovered:\n")
for cid, label in label_map.items():
    print(f"  -> Mathematical Cluster {cid} is actually the medical subtype: {label}")
    
# Add the labels back to our patient spreadsheet
df_clean['Cluster_ID'] = cluster_labels
df_clean['Phenotype'] = df_clean['Cluster_ID'].map(label_map)

## 5. The Results (The Patient Summary)
Let's look at the "average" patient inside each of the 4 buckets the AI created. This is exactly what a doctor looks at to prescribe medicine.

In [ ]:
summary_data = []

for phenotype in ['SIRD', 'SIDD', 'MOD', 'MARD']:
    subset = df_clean[df_clean['Phenotype'] == phenotype]
    if len(subset) == 0: continue
        
    n = len(subset)
    pct = (n / len(df_clean)) * 100
    
    # Get the "Median" (The middle patient) for each metric
    bmi = subset['bmi'].median()
    age = subset['age'].median()
    hdl = subset['hdl'].median() # Good cholesterol
    tg = subset['triglycerides'].median() # Bad fat in blood
    
    char_str = f"Age: {age:.1f} | BMI: {bmi:.1f} | Good Chol(HDL): {hdl:.1f} | Fat(TG): {tg:.1f}"
    
    # Add medical explanations so the reader understands what it means
    if phenotype == 'MOD':
        explain = "Obesity-Driven. High BMI. Needs Weight Loss."
    elif phenotype == 'MARD':
        explain = "Age-Driven. Thinner, older. Mild disease."
    elif phenotype == 'SIDD':
        explain = "Lipid-Driven. Very high bad cholesterol. Needs Statins."
    elif phenotype == 'SIRD':
        explain = "Insulin-Resistant. Most severe. High fat, very low good cholesterol."
        
    summary_data.append({
        "Medical Subtype": phenotype,
        "Total Patients": f"{n} ({pct:.1f}%)",
        "The 'Average' Patient looks like:": char_str,
        "Doctor's Translation": explain
    })

summary_df = pd.DataFrame(summary_data)
display(summary_df.style.set_properties(**{'text-align': 'left', 'padding': '10px'}))

## 6. Visualizing the Disease (The PCA Scatter Plot)

Humans can't visualize 8 dimensions of biology at once (Age, BMI, HDL, LDL, etc.). 
A mathematical trick called **PCA (Principal Component Analysis)** squishes those 8 dimensions down into a flat 2D map so we can look at it on a computer screen.

Every dot below is a patient. Notice how the AI naturally grouped them into distinct "continents" of sickness!

In [ ]:
# Squish 8 biological dimensions down to 2 dimensions (X and Y axis) so we can draw it.
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

plt.figure(figsize=(10, 8))
# Give each disease a unique color
colors = {'SIRD': '#e74c3c', 'SIDD': '#c0392b', 'MOD': '#f39c12', 'MARD': '#27ae60'}

for phenotype in ['SIRD', 'SIDD', 'MOD', 'MARD']:
    mask = df_clean['Phenotype'] == phenotype
    plt.scatter(X_pca[mask, 0], X_pca[mask, 1], 
                c=colors.get(phenotype, '#333'), 
                label=phenotype, alpha=0.6, s=50)

plt.xlabel(f'Biological Dimension 1 (Captures {pca.explained_variance_ratio_[0]*100:.1f}% of variance)')
plt.ylabel(f'Biological Dimension 2 (Captures {pca.explained_variance_ratio_[1]*100:.1f}% of variance)')
plt.title('The Diabetes Spectrum (DIANA AI)\nEvery dot is a patient, mathematically grouped by their biology')
plt.legend(title="Medical Subtype")
plt.grid(alpha=0.3)
plt.show()

## 7. The Disease "Fingerprint" (Radar Chart)

A Radar Chart (or Spider Plot) is a great way to see the unique "shape" of each disease subtype. 

We calculate the average standardized value for each lab test. 
* If the shape spikes toward **BMI**, that subtype is driven by Obesity (MOD).
* If the shape spikes toward **LDL/Triglycerides**, that subtype is driven by toxic fats in the blood (SIDD or SIRD).

In [ ]:
import math

# Calculate the mean Z-scores for each cluster to plot the radar chart
cluster_means = df_clean.groupby('Phenotype')[CLUSTER_FEATURES].mean()
scaler_means = pd.DataFrame(scaler.transform(cluster_means), columns=CLUSTER_FEATURES, index=cluster_means.index)

# Setup the radar chart
categories = CLUSTER_FEATURES
N = len(categories)
angles = [n / float(N) * 2 * math.pi for n in range(N)]
angles += angles[:1]  # Complete the circle

fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))

for phenotype in ['SIRD', 'SIDD', 'MOD', 'MARD']:
    if phenotype not in scaler_means.index: continue
    values = scaler_means.loc[phenotype].values.flatten().tolist()
    values += values[:1]
    ax.plot(angles, values, linewidth=2, linestyle='solid', label=phenotype, c=colors.get(phenotype, '#333'))
    ax.fill(angles, values, alpha=0.1, c=colors.get(phenotype, '#333'))

plt.xticks(angles[:-1], categories)
ax.set_rlabel_position(0)
plt.title('Biological Fingerprints of the 4 Diabetes Subtypes', size=15, y=1.1)
plt.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1))
plt.show()

## 8. Deep Dive: Lab Tests by Subtype (Boxplots)

Let's look at the actual distribution of the real-world lab tests (not standardized math numbers) across the 4 subtypes. 

Notice how **MOD** dominates the BMI chart, while **SIDD/SIRD** dominate the cholesterol charts.

In [ ]:
features_to_plot = ['age', 'bmi', 'ldl', 'triglycerides']
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for i, feature in enumerate(features_to_plot):
    sns.boxplot(x='Phenotype', y=feature, data=df_clean, 
                order=['SIRD', 'SIDD', 'MOD', 'MARD'], 
                palette=colors, ax=axes[i])
    axes[i].set_title(f'Distribution of {feature.upper()}')
    axes[i].set_ylabel(feature.upper())
    axes[i].set_xlabel('')

plt.tight_layout()
plt.show()